# Load the data set

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
df = pd.read_csv("../data/REAL_DATA.csv")

In [3]:
print(df.shape)

(71205, 9)


In [4]:
print(df.columns)

Index(['index', 'store_ID', 'day_of_week', 'date', 'nb_customers_on_day',
       'open', 'promotion', 'state_holiday', 'school_holiday'],
      dtype='str')


In [5]:
print(df.dtypes)

index                  int64
store_ID               int64
day_of_week            int64
date                     str
nb_customers_on_day    int64
open                   int64
promotion              int64
state_holiday            str
school_holiday         int64
dtype: object


In [6]:
print(df.isna())

       index  store_ID  day_of_week   date  nb_customers_on_day   open  \
0      False     False        False  False                False  False   
1      False     False        False  False                False  False   
2      False     False        False  False                False  False   
3      False     False        False  False                False  False   
4      False     False        False  False                False  False   
...      ...       ...          ...    ...                  ...    ...   
71200  False     False        False  False                False  False   
71201  False     False        False  False                False  False   
71202  False     False        False  False                False  False   
71203  False     False        False  False                False  False   
71204  False     False        False  False                False  False   

       promotion  state_holiday  school_holiday  
0          False          False           False  
1          

In [7]:
df.head()

,index,store_ID,day_of_week,date,nb_customers_on_day,open,promotion,state_holiday,school_holiday
0,272371,415,7,01/03/2015,0,0,0,0,0
1,558468,27,7,29/12/2013,0,0,0,0,0
2,76950,404,3,19/03/2014,657,1,1,0,0
3,77556,683,2,29/01/2013,862,1,0,0,0
4,456344,920,3,19/03/2014,591,1,1,0,0


In [8]:
print(df.isna().sum())

index                  0
store_ID               0
day_of_week            0
date                   0
nb_customers_on_day    0
open                   0
promotion              0
state_holiday          0
school_holiday         0
dtype: int64


In [9]:
print("state holiday:", df["state_holiday"].unique())
print("number of customers per day:", df["nb_customers_on_day"].unique())
print("open:", df["open"].unique())

state holiday: <StringArray>
['0', 'a', 'b', 'c']
Length: 4, dtype: str
number of customers per day: [   0  657  862 ... 1848 3366 2321]
open: [0 1]


In [10]:
df[df["nb_customers_on_day"] == 0]

,index,store_ID,day_of_week,date,nb_customers_on_day,open,promotion,state_holiday,school_holiday
0,272371,415,7,01/03/2015,0,0,0,0,0
1,558468,27,7,29/12/2013,0,0,0,0,0
8,162710,756,4,04/06/2015,0,0,1,a,0
12,44740,897,7,20/04/2014,0,0,0,0,0
20,274598,1103,7,15/02/2015,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...
71170,587208,346,7,27/01/2013,0,0,0,0,0
71193,138958,970,7,14/04/2013,0,0,0,0,0
71199,620934,922,4,03/10/2013,0,0,0,a,0
71200,59062,441,7,26/10/2014,0,0,0,0,0


# Find the anomalous row

# Data Cleaning

### Drop the row identifier

In [11]:
df.head()

,index,store_ID,day_of_week,date,nb_customers_on_day,open,promotion,state_holiday,school_holiday
0,272371,415,7,01/03/2015,0,0,0,0,0
1,558468,27,7,29/12/2013,0,0,0,0,0
2,76950,404,3,19/03/2014,657,1,1,0,0
3,77556,683,2,29/01/2013,862,1,0,0,0
4,456344,920,3,19/03/2014,591,1,1,0,0


In [12]:
# Drop index column (same as Unnamed: 0 in training)
df = df.drop(columns=['index'])
df.head()

,store_ID,day_of_week,date,nb_customers_on_day,open,promotion,state_holiday,school_holiday
0,415,7,01/03/2015,0,0,0,0,0
1,27,7,29/12/2013,0,0,0,0,0
2,404,3,19/03/2014,657,1,1,0,0
3,683,2,29/01/2013,862,1,0,0,0
4,920,3,19/03/2014,591,1,1,0,0


### Fix `date` data types , extract useful features, drop the original `date` column

In [13]:
# Check what the issue is
print(df.dtypes)
print(df.isna().sum())
print(df['state_holiday'].unique())

store_ID               int64
day_of_week            int64
date                     str
nb_customers_on_day    int64
open                   int64
promotion              int64
state_holiday            str
school_holiday         int64
dtype: object
store_ID               0
day_of_week            0
date                   0
nb_customers_on_day    0
open                   0
promotion              0
state_holiday          0
school_holiday         0
dtype: int64
<StringArray>
['0', 'a', 'b', 'c']
Length: 4, dtype: str


In [15]:
df['state_holiday'] = df['state_holiday'].astype(str)

KeyError: 'state_holiday'

### for column `state_holiday ` Creates a binary column for each category, drops holiday_0 to avoid the dummy variable trap (multicollinearity), 

In [14]:
df = pd.get_dummies(df, columns=['state_holiday'], prefix='holiday', drop_first=True)
print(df.head())

   store_ID  day_of_week        date  nb_customers_on_day  open  promotion  \
0       415            7  01/03/2015                    0     0          0   
1        27            7  29/12/2013                    0     0          0   
2       404            3  19/03/2014                  657     1          1   
3       683            2  29/01/2013                  862     1          0   
4       920            3  19/03/2014                  591     1          1   

   school_holiday  holiday_a  holiday_b  holiday_c  
0               0      False      False      False  
1               0      False      False      False  
2               0      False      False      False  
3               0      False      False      False  
4               0      False      False      False  


# Model 1: linear regression

define the features and target

train and test split

train linear regression

evaluate